In [ ]:
import pandas as pd

## After Data Exploration found soem unwanted columns and some misssing values

In [ ]:
df = pd.read_csv('final_data.csv')
# Dropping unwanted columns
df.drop('OrderValue', axis=1, inplace=True)
df.drop('OrderID', axis=1, inplace=True)
df.drop('CustomerID', axis=1, inplace=True)
df.drop('ProductID', axis=1, inplace=True)
df.drop('SignupDate', axis=1, inplace=True)

In [ ]:
# Filling the missing values
df['Age'] = df['Age'].fillna(df['Age'].median()).round().astype('int32')
df['City'] = df['City'].fillna(df['City'].mode()[0])

In [ ]:
# Fixing incorrect data types
numerical_cols = ['Quantity', 'Discount', 'Age', 'UnitPrice', 'Sales']
categorical_cols = ['PaymentMethod', 'Status', 'City', 'CustomerSegment', 'ProductName', 'Category']
date_cols = ['OrderDate']

for col in numerical_cols:
    df[col] = df[col].astype('int32')

for col in categorical_cols:
    df[col] = df[col].astype('category')

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [ ]:
df.info()

In [ ]:
df.head()

## Before Training the model we need to encode and scale the columns as per need
## Also extracting Date, Day and Month from date and droping the OrderDate Column

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df['OrderDay'] = df['OrderDate'].dt.day
df['OrderMonth'] = df['OrderDate'].dt.month
df['OrderYear'] = df['OrderDate'].dt.year

df.drop('OrderDate', axis=1, inplace=True)

numeric_features = ['Quantity', 'Discount', 'Age', 'UnitPrice', 'OrderDay', 'OrderMonth', 'OrderYear']

categorical_features = ['PaymentMethod', 'City', 'CustomerSegment', 'ProductName', 'Category']

feature = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

feature

## Task 1 --> Linear Regression Algorithm

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

x = df.drop('Sales', axis=1)
y = df['Sales']

lr = LinearRegression()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train = feature.fit_transform(x_train)
x_test = feature.transform(x_test)

# Training Linear Regression model
lr.fit(x_train, y_train)

y_predict = lr.predict(x_test)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_test, y=y_predict)
plt.show()

## Task 2 --> Regression Evaluation Metrics
#### MAE
#### MSE
#### RMSE

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

# Mean Absolute Error
mae = mean_absolute_error(y_test, y_predict)
# Mean Squared Error
mse = mean_squared_error(y_test, y_predict)
# Root Mean Squared Error
rmse = root_mean_squared_error(y_test, y_predict)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)

## Task 3 --> Logistic Regression -> Classification

In [ ]:
X = df.drop('Status', axis=1)
Y = df['Status']
Y.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

X_train = feature.fit_transform(X_train)
X_test = feature.transform(X_test)

lr = LogisticRegression()
lr.fit(X_train, Y_train)
y_pred = lr.predict(X_test)

accuracy_lr = accuracy_score(Y_test, y_pred)
accuracy_lr

## Task 4 --> Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

X_train_nb = X_train.toarray()
X_test_nb = X_test.toarray()

nb.fit(X_train_nb, Y_train)

y_pred_nb = nb.predict(X_test_nb)

accuracy_nb = accuracy_score(Y_test, y_pred_nb)

print("Logistic Regression Accuracy:", accuracy_lr)
print("Naive Bayes Accuracy:", accuracy_nb)

## Task 5 --> KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

k_values = [3, 5, 7]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, Y_train)
    y_pred_knn = knn.predict(X_test)

    accuracy_knn = accuracy_score(Y_test, y_pred_knn)
    print(f'For k = {k}, accuracy score is {accuracy_knn}')

## Task 6 --> Metrics for Classification

#### Logistic Regression

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(Y_test, y_pred))
print("Precision:", precision_score(Y_test, y_pred, average='weighted'))
print("Recall:", recall_score(Y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(Y_test, y_pred, average='weighted'))

print("\nClassification Report:")
print(classification_report(Y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(Y_test, y_pred))

#### Naive Bayes Classifier

In [ ]:
print("Accuracy:", accuracy_score(Y_test, y_pred_nb))
print("Precision:", precision_score(Y_test, y_pred_nb, average='weighted'))
print("Recall:", recall_score(Y_test, y_pred_nb, average='weighted'))
print("F1 Score:", f1_score(Y_test, y_pred_nb, average='weighted'))

print("\nClassification Report:")
print(classification_report(Y_test, y_pred_nb))

print("Confusion Matrix:")
print(confusion_matrix(Y_test, y_pred_nb))

#### KNN metrics

In [ ]:
print("Accuracy:", accuracy_score(Y_test, y_pred_knn))
print("Precision:", precision_score(Y_test, y_pred_knn, average='weighted'))
print("Recall:", recall_score(Y_test, y_pred_knn, average='weighted'))
print("F1 Score:", f1_score(Y_test, y_pred_knn, average='weighted'))

print("\nClassification Report:")
print(classification_report(Y_test, y_pred_knn))

print("Confusion Matrix:")
print(confusion_matrix(Y_test, y_pred_knn))

## Task 7 --> Overfitting and UnderFitting using KNeighborsClassifier

In [156]:
knn_underfit = KNeighborsClassifier(n_neighbors=50)

knn_underfit.fit(X_train, y_train)

# Predictions
y_train_pred_underfit = knn_underfit.predict(X_train)
y_test_pred_underfit = knn_underfit.predict(X_test)

# Accuracy
train_accuracy_underfit = accuracy_score(y_train, y_train_pred_underfit)
test_accuracy_underfit = accuracy_score(y_test, y_test_pred_underfit)

print("Training Accuracy:", train_accuracy_underfit)
print("Testing Accuracy :", test_accuracy_underfit)

Training Accuracy: 0.5347537902836681
Testing Accuracy : 0.4727272727272727


In [155]:
knn_overfit = KNeighborsClassifier(n_neighbors=1)

knn_overfit.fit(X_train, y_train)

# Predictions
y_train_pred_overfit = knn_overfit.predict(X_train)
y_test_pred_overfit = knn_overfit.predict(X_test)

# Accuracy
train_accuracy_overfit = accuracy_score(y_train, y_train_pred_overfit)
test_accuracy_overfit = accuracy_score(y_test, y_test_pred_overfit)

print("Training Accuracy:", train_accuracy_overfit)
print("Testing Accuracy :", test_accuracy_overfit)

Training Accuracy: 1.0
Testing Accuracy : 0.43067546978161503


## Task 8 --> Bias and Variance

### bias in ML models
Bias is the error caused by a model making overly simple assumptions about the data. 
A high-bias model is generally too simple and can lead to underfitting.

### variance
Variance is the sensitivity of a model to changes in the training data. 
A high-variance model may learn the training data too closely, including noise, which can lead to overfitting.

### bias and variance relate to underfitting and overfitting?

- High Bias → Underfitting
- High Variance → Overfitting
- Balanced Bias and Variance → Better Generalization

Underfitting occurs when the model is too simple to learn the important patterns.
Overfitting occurs when the model is too complex and learns the training data too closely.

### reduce overfitting
Overfitting can be reduced by:

1. Reducing model complexity
2. Using more training data
3. Applying regularization
4. Using cross-validation
5. Removing irrelevant features
6. Applying pruning in Decision Trees